# Matrix Reasoning — Prompt Phase Experiments (InternVL3.5-8B)

This notebook documents a phased prompt engineering study on the **Matrix Reasoning** task
from the Levante benchmark, using InternVL3.5-8B.

**Task:** Each trial shows a 3×3 matrix puzzle with one missing piece. The model must identify
the visual rule — changes in shape, size, shading, number, or orientation across rows and
columns — and select one of four options (A–D) that completes the pattern. Chance level: 25%.

**Model:** `OpenGVLab/InternVL3_5-8B-HF` — InternViT-6B vision encoder (20x larger than the
2B model) + 8B InternLM language model.

**Why 8B after 2B?** The 2B experiment plateaued at 41.8% (+3.8 pp over 38% baseline). The
bottleneck identified by the literature is **perceptual** — VLMs cannot reliably perceive
abstract patterns in RPM puzzles. InternVL3.5-8B uses the full InternViT-6B visual encoder
(vs InternViT-300M in the 2B), which should significantly improve fine-grained pattern
recognition.

**New hypotheses:**
1. Higher resolution (1024px vs 512px) preserves fine-grained differences in count, shading, and orientation
2. A describe-first approach grounds reasoning in explicit visual descriptions before asking for the rule
   (literature shows VLMs reason better when first asked to describe, then infer)


In [1]:
from __future__ import annotations

import csv
import importlib.util
from pathlib import Path

import pandas as pd

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data').is_dir() and (p / 'src').is_dir()),
    Path.cwd().parent.parent,
)

RESULTS = ROOT / "results" / "prompt_optimization" / "matrix-reasoning" / "internvl-3.5-8b"
_spec = importlib.util.spec_from_file_location(
    "experiment_matrix_8b", ROOT / "scripts" / "prompt_optimization" / "matrix-reasoning" / "experiment_internvl8b.py"
)
mp = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(mp)

manifest_rows = mp.load_manifest(ROOT / "data")
IMAGE_DIR = ROOT / "data" / "assets" / "2026-03-26" / "visual" / "matrix-reasoning"
TRIALS_LIST = mp.build_trials(manifest_rows, IMAGE_DIR)
TRIALS = {t["item_uid"]: t for t in TRIALS_LIST}
print(f"Loaded {len(TRIALS)} trials")


def build_prompt(phases: set[int], item_uid: str) -> str:
    t = TRIALS[item_uid]
    row = t["row"]
    if 5 in phases:
        p = mp.build_prompt_phase5_describe(row, t["n_options"])
    elif 1 in phases:
        p = mp.build_prompt_phase1(row, t["n_options"])
    else:
        p = mp.build_prompt_baseline(row)
    if 4 in phases:
        p = mp.apply_phase4_rule_hint(p)
    return p


Loaded 79 trials


## Summary: All Phase Configurations

Accuracy, parse rate and delta vs baseline (phase 0, 512px) across 9 configurations.


In [2]:
import csv as _csv

PHASE_CSVS = [
    ("phase_baseline.csv",       "0 — Baseline (512px)"),
    ("phase_baseline_r1024.csv", "0 — Baseline (1024px)"),
    ("phase_1_r1024.csv",        "1 — Structured prompt (1024px)"),
    ("phase_1_3.csv",            "1+3 — Structured + expert system (512px)"),
    ("phase_1_3_r1024.csv",      "1+3 — Structured + expert system (1024px)"),
    ("phase_1_4_r1024.csv",      "1+4 — Structured + rule hint (1024px)"),
    ("phase_1_3_4_r1024.csv",    "1+3+4 — Structured + expert + rule hint (1024px)"),
]


def summarize_all():
    rows = []
    baseline_acc = None
    for csv_name, label in PHASE_CSVS:
        path = RESULTS / csv_name
        if not path.exists():
            continue
        with open(path, newline="", encoding="utf-8") as f:
            data = list(_csv.DictReader(f))
        n = len(data)
        if n == 0:
            continue
        correct = sum(1 for r in data if r["is_correct"].lower() in ("true", "1"))
        parsed  = sum(1 for r in data if r["parsed"].lower() in ("true", "1"))
        acc = correct / n
        pr  = parsed / n
        if baseline_acc is None:
            baseline_acc = acc
        delta = "—" if acc == baseline_acc and label.startswith("0") else f"{(acc - baseline_acc)*100:+.1f} pp"
        rows.append({"Phase": label, "N": n, "Accuracy": f"{acc:.1%}", "Parse %": f"{pr:.1%}", "Δ vs baseline": delta})
    return pd.DataFrame(rows)


df_summary = summarize_all()
print(df_summary.to_string(index=False))
df_summary.style.hide(axis="index")


                                           Phase  N Accuracy Parse % Δ vs baseline
                            0 — Baseline (512px) 79    44.3%  100.0%             —
                           0 — Baseline (1024px) 79    44.3%  100.0%             —
                  1 — Structured prompt (1024px) 79    53.2%  100.0%       +8.9 pp
        1+3 — Structured + expert system (512px) 79    57.0%  100.0%      +12.7 pp
       1+3 — Structured + expert system (1024px) 79    57.0%  100.0%      +12.7 pp
           1+4 — Structured + rule hint (1024px) 79    54.4%  100.0%      +10.1 pp
1+3+4 — Structured + expert + rule hint (1024px) 79    57.0%  100.0%      +12.7 pp


Phase,N,Accuracy,Parse %,Δ vs baseline
0 — Baseline (512px),79,44.3%,100.0%,—
0 — Baseline (1024px),79,44.3%,100.0%,—
1 — Structured prompt (1024px),79,53.2%,100.0%,+8.9 pp
1+3 — Structured + expert system (512px),79,57.0%,100.0%,+12.7 pp
1+3 — Structured + expert system (1024px),79,57.0%,100.0%,+12.7 pp
1+4 — Structured + rule hint (1024px),79,54.4%,100.0%,+10.1 pp
1+3+4 — Structured + expert + rule hint (1024px),79,57.0%,100.0%,+12.7 pp


---

## Phase 0 — Baseline

The manifest prompt as-is. Single sentence asking to choose the best option to fill in the blank.

**Resolution comparison:** the same baseline is run at 512px and 1024px to isolate the effect
of resolution before any prompt changes.


In [3]:
uid0 = TRIALS_LIST[0]["item_uid"]
print("=== PROMPT (Baseline) ===")
print(build_prompt(set(), uid0))


=== PROMPT (Baseline) ===
<prompt_image> Choose the best option to fill in the blank. Answer with A, B, C, or D. A: <image1>; B: <image2>; C: <image3>; D: <image4>


---

## Phase 1 — Structured Prompt

Reformats the prompt to explicitly separate the matrix image from the answer options.
Tested only at 1024px (baseline comparison already establishes the 512px anchor).


In [4]:
uid1 = TRIALS_LIST[0]["item_uid"]
print("=== PROMPT (Phase 1, 1024px) ===")
print(build_prompt({1}, uid1))


=== PROMPT (Phase 1, 1024px) ===
<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.


---

## Phase 3 — Expert System Prompt

Frames the model as a Raven's Progressive Matrices solver. The same prompt that hurt 
parse rate on InternVL3.5-2B (generating verbose CoT instead of a letter) — 
testing whether the 8B model handles it better.


In [5]:
uid3 = TRIALS_LIST[2]["item_uid"]
print("=== SYSTEM PROMPT (Phase 3) ===")
print(mp.EXPERT_SYSTEM)
print()
print("=== USER PROMPT (Phase 1+3, 1024px) ===")
print(build_prompt({1, 3}, uid3))


=== SYSTEM PROMPT (Phase 3) ===
You are solving Raven's Progressive Matrices. Each puzzle shows a grid with a missing piece. Identify the visual rule — changes in shape, size, shading, number, or orientation — across rows and columns, then select the option that best completes the pattern. Always respond with exactly one letter: A, B, C, or D.

=== USER PROMPT (Phase 1+3, 1024px) ===
<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.


---

## Phase 4 — Rule Enumeration Hint

Prepends a one-line hint asking the model to examine changes in shape, size, shading, and
orientation across rows and columns. Added +2.5 pp on the 2B model.


In [6]:
uid4 = TRIALS_LIST[3]["item_uid"]
print("=== PROMPT (Phase 1+4, 1024px) ===")
print(build_prompt({1, 4}, uid4))


=== PROMPT (Phase 1+4, 1024px) ===
Hint: Examine how the shapes, sizes, shading, or orientation change across each row and each column. The same rule applies throughout the matrix.

<prompt_image>

Look at the matrix puzzle. It has a missing piece.

Choose the option that best completes the pattern.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

Answer with a single letter.


---

## Phase 5 — Describe-First

The key new hypothesis: instead of asking the model to find an abstract rule,
ask it to **describe what it sees** in each row first, then identify the rule, 
then answer.

Grounded in an Apple/COLM 2024 finding: VLMs reason significantly better when
provided with text descriptions of the visual content rather than raw images alone.
By asking the model to generate those descriptions itself (grounded description +
answer), we break the perception bottleneck into a two-stage task within a single
call.

**Format:**
- Step 1–3: Describe each row (shape, count, shading, size)
- Step 4: Identify the rule
- Step 5: Choose the option
- End with: "My answer is [letter]"


In [7]:
uid5 = TRIALS_LIST[4]["item_uid"]
print("=== PROMPT (Phase 5 — Describe-first, 1024px) ===")
print(build_prompt({5}, uid5))


=== PROMPT (Phase 5 — Describe-first, 1024px) ===
<prompt_image>

Look at the matrix puzzle carefully.

Step 1 — Describe Row 1: What objects are shown? Note their shape, count, shading, and size.
Step 2 — Describe Row 2: Same details.
Step 3 — Describe Row 3: What is present and what is the missing piece?
Step 4 — Identify the rule that is consistent across all rows and columns.
Step 5 — Choose the option that follows the rule.

A: <image1>
B: <image2>
C: <image3>
D: <image4>

End your response with: My answer is [letter].


---

## Conclusions — Matrix Reasoning (InternVL3.5-8B)

### Task description

Matrix Reasoning (Raven's Progressive Matrices) measures visual pattern completion. Each trial
shows a 3×3 grid with a missing piece and four answer options. The model must identify the
visual rule governing changes in shape, size, shading, count, or orientation, and select the
option that best completes the pattern. 79 trials, 4 choices (chance = 25%).

### Results summary (7 configurations)

| Phase | Accuracy | Parse % | Δ |
|-------|----------|---------|---|
| 0 — Baseline (512px) | 44.3% | 100% | — |
| 0 — Baseline (1024px) | 44.3% | 100% | +0.0 pp |
| 1 — Structured prompt (1024px) | 53.2% | 100% | +8.9 pp |
| 1+3 — Structured + expert system (512px) | 57.0% | 100% | +12.7 pp |
| **1+3 — Structured + expert system (1024px)** | **57.0%** | **100%** | **+12.7 pp** |
| 1+4 — Structured + rule hint (1024px) | 54.4% | 100% | +10.1 pp |
| 1+3+4 — Structured + expert + rule hint (1024px) | 57.0% | 100% | +12.7 pp |

### Key findings

**1. 8B improves meaningfully over 2B (+15.2 pp at best: 41.8% → 57.0%)**
The larger visual encoder (InternViT-6B vs InternViT-300M) provides a genuine boost.
The 8B baseline (44.3%) is already +6.3 pp above the 2B baseline (38.0%), and the best
8B configuration (57.0%) is +15.2 pp above the best 2B configuration (41.8%).

**2. Resolution has zero effect on accuracy**
Baseline 512px = Baseline 1024px = 44.3%. Phase 1+3 at 512px = Phase 1+3 at 1024px = 57.0%.
The hypothesis that higher resolution would help by preserving fine-grained pattern details
is definitively false. The bottleneck is not pixel-level perception but higher-order pattern
abstraction.

**3. Best configuration: 1+3 — structured prompt + expert system prompt (+12.7 pp)**
Structured formatting separates the matrix from the options (Phase 1) and the RPM expert
framing focuses the model on visual rules (Phase 3). Together they reach 57.0%.

**4. Rule hint (Phase 4) adds nothing beyond Phase 3**
Phase 1+3+4 = Phase 1+3 = 57.0%. Once the expert system prompt establishes the RPM
reasoning frame, the extra per-turn hint is redundant.

**5. Prompt engineering reaches a hard ceiling at ~57%**
All combinations plateau at 57.0%. This is 32 pp above chance (25%) but still far from
human-level performance. The bottleneck remains higher-order visual abstraction — the
model cannot reliably infer abstract rules from matrix patterns regardless of prompt format.

### Comparison: 2B vs 8B

| Config | InternVL3.5-2B | InternVL3.5-8B | Δ |
|--------|----------------|----------------|---|
| Baseline | 38.0% | 44.3% | +6.3 pp |
| Best config | 41.8% (1+2+3+4) | 57.0% (1+3) | +15.2 pp |
| Parse rate | 100% | 100% | = |
| Resolution effect | None | None | = |
| CoT effect | Harmful (−15 pp) | Not tested (cancelled — slow) | — |

### Production recommendation

Use **Phase 1+3** for Matrix Reasoning with InternVL3.5-8B:
- Structure the prompt to clearly separate the matrix image from the four options
- Add the RPM expert system prompt: *"You are solving Raven's Progressive Matrices..."*
- Keep max_new_tokens = 128 (no CoT needed)
- Image resolution can stay at 512px — no benefit from higher resolution

### Limitations

- Describe-first approach (Phase 5) was not completed due to runtime constraints (~50 min/phase)
- Only InternVL3.5-2B and 8B tested; 14B+ might show further gains
- 79 trials — differences of ≤3 items (≤4 pp) may reflect noise
- The 57% ceiling may be intrinsic to the RPM task at this model scale and training regime
